# KTB RAG 예선 — 팀 8 평가기

팀별 답변 파일을 골드셋과 맞대어 **총점(0~100)** 을 매기고, 교차평가 결과를
`eval_8.json` 으로 저장한다.

## 채점식

```
총점 = 0.20 × (MRR × 100)
     + 0.30 × (키팩트 F1 × 100)
     + 0.50 × (판정 점수 0~100)
```

| 지표 | 무엇을 보나 | 어떻게 재나 |
|---|---|---|
| **MRR** | 근거 조항을 몇 번째로 맞혔나 | 정답 조항이 처음 나온 순위의 역수. `gold_articles` 중 하나만 맞아도 정답 |
| **키팩트 F1** | 답변 내용이 정답과 얼마나 겹치나 | 답변 전체를 토큰 **집합**으로 보고 `key_facts` 전체와 F1 |
| **판정** | 사람이 읽었을 때 좋은 답변인가 | **Gemini 3.5 Flash** 가 accuracy·grounding·completeness·clarity 4축을 0~10으로 채점, 가중합 |

판정 4축 가중치는 `accuracy 0.40 / grounding 0.25 / completeness 0.20 / clarity 0.15` —
운영진 기준(정확성·근거성·완결성·명료성)을 그대로 따른다.

운영진 지침: **"Gemini 3.5 Flash로 공개 10문항과 비공개 30문항을 같은 코드로 처리하고,
익명 5팀 모두에 같은 기준을 적용해야 합니다."** 이 노트북은 판정 축을 실제로 Gemini
3.5 Flash API로 채점한다(5b, 9절). 규칙 기반 대리 심판(`judge_rule_based`, 5절)은
API 키 없이 도는 자체 검증(8절)과, Gemini 호출이 끝내 실패한 문항의 폴백으로만 쓴다.

## 골드셋에서 읽는 필드

`questions[].id`, `questions[].question`,
`questions[].gold_articles[].{doc, article, citation}`, `questions[].key_facts` — 이 넷뿐이다.

`ptype`·`difficulty` 같은 추가 필드는 **있어도 쓰지 않고 없어도 돈다.**
문항 id와 문항 수도 코드에 박지 않고 입력 파일에서 읽는다. 그래서 공개 10문항과
비공개 30문항에 같은 노트북을 그대로 쓴다.

## 이 노트북의 구성

| 절 | 내용 |
|---|---|
| 1~4 | 정규화, 입력 로더, MRR, 키팩트 F1 — 결정론적 지표 |
| 5 | 판정 4축 규칙 기반 대리 심판 — **자체 검증·폴백 전용**, 실채점 아님 |
| 5b | 판정 4축 **Gemini 3.5 Flash 실채점** — 실제 제출에 쓰는 것 |
| 6~7 | 한 팀 채점 → 교차평가 (BLIND01~05, status는 항상 completed) |
| 7b | 제출 파일 형식 검사 — 운영진 검사기와 동일 기준 |
| 8 | 자체 검증 (API 키 불필요, 5절 규칙 기반으로 배선만 확인) |
| 9 | Gemini API 키 설정 + 1문항 스모크 테스트 |
| 10 | 실행 — 업로드 → `eval_8.json` 생성 → 7b로 즉시 `[제출 가능]` 확인 |
| 11 | 문항별 들여다보기 |

> 다른 채점 방식을 쓰고 싶으면 `evaluate_team(..., judge_fn=내함수)` 로 갈아 끼우면 된다.
> `judge_fn(question, answer, key_facts, citations, retrieved, gold_keys)` 가
> `({축: 0~10점}, {축: 사유})` 를 돌려주면 된다. 운영진도 "팀이 중요하다고 판단한
> 기준이 있으면 새로 설계해도 된다"고 했으므로, 축 구성 자체를 바꾸는 것도 자유다 —
> 다만 Gemini 3.5 Flash로 공개·비공개 문항과 5팀 전원을 같은 코드로 처리한다는
> 제약은 유지해야 한다.

In [ ]:
# -*- coding: utf-8 -*-
# 표준 라이브러리만 쓴다. 새 Colab 런타임에서 설치 없이 바로 돈다.
# (5b의 Gemini 판정만 예외로 google-genai가 필요하며, 그 설치는 해당 셀에서 한다.)
from __future__ import annotations

import json
import math
import re
import time
import unicodedata
from collections import Counter
import os  # 노트북 9·10절 전용(경로 표시, API 키 환경변수) — 코어 자체는 안 씀


# =====================================================================================
# 0. 채점식 상수
# =====================================================================================
# 축 가중치: 총점 = 0.2*MRR(100) + 0.3*keyfact_f1(100) + 0.5*judge(100)
WEIGHTS = {"mrr": 0.20, "keyfact": 0.30, "judge": 0.50}

# 판정(judge) 4축 가중치. 축 점수는 0~10, judge_total = 10 * Σ w*s
JUDGE_AXIS_WEIGHTS = {
    "accuracy": 0.40,
    "grounding": 0.25,
    "completeness": 0.20,
    "clarity": 0.15,
}

# 근거 표기에 쓰이는 공식 문서명 4종. 채점에 직접 쓰지는 않고(골드셋의 doc이 기준),
# 답변 파일에 낯선 문서명이 나왔을 때 알려 주는 용도로만 둔다.
OFFICIAL_DOCUMENT_NAMES = (
    "카카오계정 약관",
    "카카오 위치정보 이용약관",
    "카카오 통합서비스약관",
    "카카오 통합 약관",
)

## 1. 정규화 — 표기가 흔들려도 같은 것은 같게 본다

남의 팀 파일을 채점하므로 표기 차이로 억울한 감점이 나면 안 된다.

- 조번호: `10`, `"10"`, `"제10조"`, `"제 10 조"` 를 모두 10으로 읽는다.
- 문서명: 공백을 지우고 비교해 `카카오 통합 약관` 과 `카카오통합약관` 을 같게 본다.
  (`카카오 통합서비스약관` 과 `카카오 통합 약관` 은 공백을 지워도 다르므로 안전하다.)
- 토큰: `\w+` 로 끊는다. 가운뎃점(`수집·이용·제공`)은 경계가 되고, `제16조` 같은
  덩어리는 한 토큰으로 남는다.

In [ ]:
# =====================================================================================
# 1. 정규화 유틸 — 표기 흔들림을 흡수한다
# =====================================================================================
def nfkc(text) -> str:
    """유니코드 정규화. 전각/반각·호환문자 차이를 없앤다."""
    if text is None:
        return ""
    if not isinstance(text, str):
        text = str(text)
    return unicodedata.normalize("NFKC", text)


def tokenize(text) -> list:
    r"""채점용 토큰화.

    `\w+` 로 끊는다. 공백·구두점·가운뎃점(·)은 경계가 되고 한글/영문/숫자 덩어리만 남는다.
    예) "제16조 제2항에 근거하여" -> ['제16조', '제2항에', '근거하여']
    """
    return re.findall(r"\w+", nfkc(text))


def normalize_doc_name(name) -> str:
    """문서명 비교용 키. 공백을 모두 없애 '카카오 통합 약관'과 '카카오통합약관'을 같게 본다.

    '카카오 통합서비스약관'과 '카카오 통합 약관'은 공백 제거 후에도 서로 다르므로 안전하다.
    """
    return re.sub(r"\s+", "", nfkc(name))


def parse_article_no(value):
    """조번호를 정수로 뽑아낸다. 3 / "3" / "제3조" / "제 3 조" 모두 3으로 본다.

    숫자를 찾지 못하면 None.
    """
    if isinstance(value, bool):
        return None
    if isinstance(value, int):
        return value
    match = re.search(r"\d+", nfkc(value))
    return int(match.group()) if match else None


def article_key(doc, article):
    """(문서명, 조번호) 비교 키. 어느 한쪽이라도 못 읽으면 None."""
    doc_key = normalize_doc_name(doc)
    no = parse_article_no(article)
    if not doc_key or no is None:
        return None
    return (doc_key, no)

## 2. 입력 읽기

골드셋은 **필수 필드가 없으면 즉시 중단**한다(내 채점 기준이 깨진 것이므로).
반대로 답변 파일은 **어떻게 깨져 있어도 중단하지 않는다** — 남의 팀 파일이고,
한 팀이 잘못 냈다고 교차평가 전체가 멈추면 안 되기 때문이다. 깨진 부분은
문제로 기록하고 해당 문항만 0점 처리한다.

In [ ]:
# =====================================================================================
# 2. 입력 로더 — 공통 필드만 읽고, 추가 필드에 의존하지 않는다
# =====================================================================================
class GoldSet:
    """골드셋 한 벌. questions[] 에서 공통 4필드만 뽑아 들고 있는다."""

    def __init__(self, questions, source=""):
        self.questions = questions          # [{id, question, gold_keys, citations, key_facts}]
        self.source = source
        self.qids = [q["id"] for q in questions]

    def __len__(self):
        return len(self.questions)


def load_goldset(path_or_payload) -> GoldSet:
    """골드셋 JSON을 읽는다.

    읽는 필드: questions[].id / .question / .gold_articles[{doc,article,citation}] / .key_facts
    그 밖의 필드(_meta, ptype, difficulty, tag ...)는 있으면 무시하고 없어도 된다.
    """
    if isinstance(path_or_payload, dict):
        payload, source = path_or_payload, "<payload>"
    else:
        source = str(path_or_payload)
        with open(path_or_payload, encoding="utf-8") as fp:
            payload = json.load(fp)

    raw_questions = payload.get("questions")
    if not isinstance(raw_questions, list) or not raw_questions:
        raise ValueError("골드셋에 questions 배열이 없습니다: {}".format(source))

    questions, seen = [], set()
    for index, item in enumerate(raw_questions, 1):
        if not isinstance(item, dict):
            raise ValueError("{}번째 문항을 읽을 수 없습니다.".format(index))

        qid = item.get("id")
        if not isinstance(qid, str) or not qid.strip():
            raise ValueError("{}번째 문항에 id가 없습니다.".format(index))
        qid = qid.strip()
        if qid in seen:
            raise ValueError("골드셋에 중복된 id가 있습니다: {}".format(qid))
        seen.add(qid)

        # gold_articles: 정답으로 인정하는 조항 목록(여러 개면 그중 하나만 맞아도 정답)
        gold_keys, citations = [], []
        for art in item.get("gold_articles") or []:
            if not isinstance(art, dict):
                continue
            key = article_key(art.get("doc"), art.get("article"))
            if key is not None:
                gold_keys.append(key)
            citation = art.get("citation")
            if isinstance(citation, str) and citation.strip():
                citations.append(citation.strip())

        key_facts = [f.strip() for f in (item.get("key_facts") or [])
                     if isinstance(f, str) and f.strip()]

        questions.append({
            "id": qid,
            "question": nfkc(item.get("question") or ""),
            "gold_keys": gold_keys,
            "citations": citations,
            "key_facts": key_facts,
        })

    return GoldSet(questions, source)


def load_answers(path_or_payload):
    """팀 답변 파일을 읽는다. 깨져 있어도 예외를 던지지 않고 문제를 기록해 돌려준다.

    반환: (answers_by_qid, info)
      answers_by_qid: {qid: {"answer": str, "retrieved": [(doc, article), ...]}}
      info: {"team":..., "blind_id":..., "problems":[...], "fatal": bool}
    """
    info = {"team": None, "blind_id": None, "problems": [], "fatal": False,
            "source": "<payload>", "n_answers": 0}

    if isinstance(path_or_payload, dict):
        payload = path_or_payload
    else:
        info["source"] = str(path_or_payload)
        try:
            with open(path_or_payload, encoding="utf-8") as fp:
                payload = json.load(fp)
        except Exception as exc:                       # 파일이 없거나 JSON이 깨진 경우
            info["problems"].append("파일을 읽을 수 없습니다: {}".format(exc))
            info["fatal"] = True
            return {}, info

    if not isinstance(payload, dict):
        info["problems"].append("최상위가 객체(JSON object)가 아닙니다.")
        info["fatal"] = True
        return {}, info

    team = payload.get("team")
    info["team"] = team.strip() if isinstance(team, str) and team.strip() else None
    blind = payload.get("blind_id")
    info["blind_id"] = blind.strip() if isinstance(blind, str) and blind.strip() else None

    raw_answers = payload.get("answers")
    if not isinstance(raw_answers, list) or not raw_answers:
        info["problems"].append("answers 배열이 비어 있거나 없습니다.")
        info["fatal"] = True
        return {}, info

    answers = {}
    for index, item in enumerate(raw_answers, 1):
        if not isinstance(item, dict):
            info["problems"].append("{}번째 답변 항목을 읽을 수 없습니다.".format(index))
            continue

        qid = item.get("qid")
        if not isinstance(qid, str) or not qid.strip():
            info["problems"].append("{}번째 답변에 qid가 없습니다.".format(index))
            continue
        qid = qid.strip()
        if qid in answers:
            info["problems"].append("qid가 중복되었습니다: {} (뒤엣것 무시)".format(qid))
            continue

        answer_text = item.get("answer")
        answer_text = answer_text if isinstance(answer_text, str) else ""

        retrieved = []
        raw_retrieved = item.get("retrieved")
        if isinstance(raw_retrieved, list):
            for pair in raw_retrieved:
                if isinstance(pair, (list, tuple)) and len(pair) == 2:
                    retrieved.append((pair[0], pair[1]))
                elif isinstance(pair, dict):           # {"doc":..,"article":..} 형태도 받아 준다
                    retrieved.append((pair.get("doc"), pair.get("article")))
        else:
            info["problems"].append("{}의 retrieved가 목록이 아닙니다.".format(qid))

        answers[qid] = {"answer": answer_text, "retrieved": retrieved}

    info["n_answers"] = len(answers)
    return answers, info

## 3. 지표 A — MRR (가중치 0.20)

`retrieved` 를 앞에서부터 훑어 정답 조항이 처음 나온 순위의 역수를 준다.
1순위면 1.0, 2순위면 0.5, 못 맞히면 0.

`gold_articles` 가 여러 개인 문항이 있다. 같은 규정이 여러 약관에 중복 수록된
경우인데, 이때는 **그중 아무거나 맞으면 정답**이다.

In [ ]:
# =====================================================================================
# 3. 지표 A — 검색 정확도 MRR (가중치 0.20)
# =====================================================================================
def score_mrr(retrieved, gold_keys):
    """retrieved를 앞에서부터 훑어 정답 조항이 처음 나오는 순위의 역수를 준다.

    gold_articles가 여러 개면 그중 아무거나 맞으면 정답으로 본다(P02처럼 동일 규정이
    세 약관에 중복 수록된 경우가 있다).

    반환: (mrr_contrib, rank)  — 못 맞히면 (0.0, 0)
    """
    if not gold_keys:
        return 0.0, 0
    gold = set(gold_keys)
    for rank, (doc, article) in enumerate(retrieved, 1):
        key = article_key(doc, article)
        if key is not None and key in gold:
            return 1.0 / rank, rank
    return 0.0, 0

## 4. 지표 B — 키팩트 F1 (가중치 0.30)

답변 전체를 토큰 **집합** 하나로 보고, `key_facts` 를 전부 이어붙인 것과 F1을 잰다.
문장별로 맞춰 보는 게 아니라 통짜 비교다. 그래서 두 방향으로 깎인다.

- 정답에 있는데 답변에 없는 토큰 → **재현율** 손해
- 답변에만 있는 군더더기 토큰 → **정밀도** 손해

즉 **빠뜨려도 깎이고 덧붙여도 깎인다.** 질문을 되풀이하거나 안 물어본 조항을
끌어오면 그만큼 점수가 내려간다.

`score_keyfact_recall` 은 팩트를 하나씩 보고 반영 여부를 세는 별도 지표다.
F1은 통짜 비교라 '어느 팩트가 통째로 빠졌는지'를 구분하지 못하는데, 이쪽이
그걸 잡아 뒤의 completeness 축 근거가 된다.

In [ ]:
# =====================================================================================
# 4. 지표 B — 키팩트 F1 (가중치 0.30)
# =====================================================================================
def score_keyfact_f1(answer, key_facts):
    """답변 전체를 토큰 '집합'으로 보고 key_facts 전체와 F1을 잰다.

    핵심: 문장 단위가 아니라 통짜 집합 비교다. 그래서
      · 정답에 있는데 답변에 없는 토큰 -> 재현율 손해
      · 답변에만 있는 군더더기 토큰   -> 정밀도 손해
    누락과 군더더기가 같은 무게로 깎인다.

    반환: (f1, precision, recall)
    """
    if not key_facts:
        return 0.0, 0.0, 0.0
    answer_tokens = set(tokenize(answer))
    gold_tokens = set(tokenize(" ".join(key_facts)))
    if not answer_tokens or not gold_tokens:
        return 0.0, 0.0, 0.0

    overlap = len(answer_tokens & gold_tokens)
    if overlap == 0:
        return 0.0, 0.0, 0.0
    precision = overlap / len(answer_tokens)
    recall = overlap / len(gold_tokens)
    return 2 * precision * recall / (precision + recall), precision, recall


# key_fact 하나가 답변에 반영됐다고 볼 임계값(그 팩트의 토큰 중 몇 %가 답변에 있는가).
# 공식 심판의 completeness 축 30건에 맞춰 정한 값이다(§보정 참고).
KEYFACT_COVER_THRESHOLD = 0.40


def score_keyfact_recall(answer, key_facts):
    """key_fact를 하나씩 보고 '반영됐는가'를 0/1로 세어 비율을 낸다.

    F1이 못 보는 것을 본다. F1은 통짜 비교라 어느 팩트가 통째로 빠졌는지 구분하지
    못하는데, 이 지표는 팩트 단위 누락을 잡아 completeness 축의 근거가 된다.

    반환: (recall, [문항별 반영여부], [팩트별 커버율])
    """
    if not key_facts:
        return 0.0, [], []
    answer_tokens = set(tokenize(answer))
    covered, ratios = [], []
    for fact in key_facts:
        fact_tokens = set(tokenize(fact))
        if not fact_tokens:
            covered.append(False)
            ratios.append(0.0)
            continue
        ratio = len(fact_tokens & answer_tokens) / len(fact_tokens)
        ratios.append(ratio)
        covered.append(ratio >= KEYFACT_COVER_THRESHOLD)
    return sum(covered) / len(covered), covered, ratios

## 5. 지표 C — 판정 4축, 규칙 기반 대리 심판 (자체 검증·폴백 전용)

**이 절의 `judge_rule_based`는 실채점에 쓰지 않는다.** 운영진 지침대로 실채점은
Gemini 3.5 Flash가 한다(5b). 이 규칙 기반 버전은 (1) API 키 없이 몇 초 안에 배선을
검증하는 8절 자체 검증과, (2) Gemini 호출이 끝내 실패한 개별 문항의 폴백으로만 쓰인다.

같은 입력에 항상 같은 점수를 내는 결정론적 함수라 자체 검증·폴백 용도로 적합하다.
축별 규칙과 임계값은 공식 심판이 매긴 축 점수 30건(3개 팀 × 공개 10문항)에 맞춰
정했다 — 그 30건에서 나온 사실이 설계를 결정했다.

| 축 | 가중치 | 무엇을 보나 | 깎는 조건 |
|---|---|---|---|
| accuracy | 0.40 | 틀리게 말한 게 있나 | 정답과 **단위는 같은데 숫자가 다름**(`6개월`을 `3개월`이라 함), 명시적 예/아니오 뒤집힘, 정답과 거의 안 겹침 |
| grounding | 0.25 | 근거가 정답 조항인가 | 정답 조항 순위(1위 10점 → 2위 8 → 3위 7 → 4위 이하 6 → 없음 2), 근거 미제시 0 |
| completeness | 0.20 | 키팩트를 빠짐없이 담았나 | 팩트별 토큰 겹침이 40% 넘으면 '반영됨'. 반영 비율을 완만한 곡선으로 점수화 |
| clarity | 0.15 | 읽히는 답변인가 | 정답 길이의 4배 초과, 같은 말 과도 반복 |

**누락은 completeness가 보고, accuracy는 '틀리게 말한 것'만 본다.** 두 축이 같은 것을
중복해서 깎지 않도록 역할을 갈랐다. 감점은 completeness에 쏠려 있었다(공식 심판
30건 중 16회 감점 중 10회). accuracy·grounding·clarity에서 길이·부정어 같은 얕은
신호로 깎아 보니 공식 점수와의 상관이 음수가 나와, 확실한 증거가 있을 때만 깎도록
좁혔다. 이 보정으로 공식 심판과의 상관이 r = 0.19 → 0.72 로 올랐다 — Gemini를 못
쓰는 상황(폴백)에서도 등급 없는 0점보다는 훨씬 신뢰할 수 있는 대체값이라는 뜻이다.

In [ ]:
# =====================================================================================
# 5. 지표 C — 판정 4축 (가중치 0.50)
# =====================================================================================
# 공식 채점은 이 축을 LLM 심판이 매긴다. 교차평가에서 팀마다 다른 심판 모델을 쓰면
# 같은 답변에 다른 점수가 나와 공정성이 깨지므로, 같은 입력이면 항상 같은 점수가
# 나오는 규칙 기반 대리 심판을 기본으로 둔다. LLM 심판은 judge_fn 인자로 갈아 끼운다.
#
# 아래 상수는 공식 심판이 매긴 축 점수 30건(3개 팀 × 공개 10문항)에 맞춰 정했다.
# 그 30건에서 관찰된 사실이 설계를 결정했다:
#   · 감점은 completeness에 쏠려 있다(16회 중 10회). 팀을 가르는 축은 사실상 여기다.
#   · accuracy·grounding·clarity는 거의 만점이다. 이 축에서 어림짐작으로 깎으면
#     실제로 상관이 음수가 됐다. 그래서 '확실한 증거가 있을 때만' 깎는다.
_COMPLETENESS_EXPONENT = 0.40         # 반영 비율 -> 점수 곡선의 지수
                                      # 임계값은 KEYFACT_COVER_THRESHOLD 하나만 쓴다

# 조번호·기간·인원처럼 틀리면 곧바로 오답이 되는 수치 단위
_NUMERIC_UNIT_RE = re.compile(r"(\d+)\s*(시간|개월|년|일|분|인|명|세|조|항|가지|번|주|회)")

# 함정 문항에서 결론이 뒤집혔는지 보는 명시적 예/아니오 표현
_EXPLICIT_NO = ("아니오", "아니요", "아닙니다", "그렇지 않습니다")
_EXPLICIT_YES = ("예,", "예.", "네,", "네.", "그렇습니다", "맞습니다")


def _numeric_pairs(text):
    """(숫자, 단위) 쌍 집합. '2시간' -> ('2','시간'), '제16조' -> ('16','조')"""
    return set(_NUMERIC_UNIT_RE.findall(nfkc(text)))


def _has_any(text, patterns):
    flat = nfkc(text)
    return any(p in flat for p in patterns)


def judge_rule_based(question, answer, key_facts, citations, retrieved, gold_keys):
    """규칙 기반 대리 심판. 4축을 각각 0~10으로 매긴다.

    반환: ({축: 점수}, {축: 사유})
    """
    axes, reasons = {}, {}
    answer_tokens = set(tokenize(answer))
    gold_tokens = set(tokenize(" ".join(key_facts)))
    has_answer = bool(answer.strip())

    # --- accuracy: 사실관계가 어긋나지 않는가 ---------------------------------------
    # 누락은 completeness가 본다. 여기서는 '틀리게 말한 것'만 본다.
    if not has_answer:
        axes["accuracy"], reasons["accuracy"] = 0.0, "답변이 비어 있음"
    else:
        score, notes = 10.0, []

        # (1) 수치 모순 — 정답과 같은 단위인데 숫자가 다르다("6개월"을 "3개월"이라 함)
        #
        # 단, 그 단위의 '맞는 값'이 답변에 함께 있으면 모순으로 보지 않는다.
        # 답변 머리에 근거를 밝히는 "(제8조) …" 같은 표기가 흔한데, 정답 키팩트에
        # '제16조'가 들어 있으면 이 8조가 틀린 조번호로 잡히기 때문이다.
        # 근거 조항이 맞는지는 grounding 축이 따로 본다.
        gold_pairs = _numeric_pairs(" ".join(key_facts))
        answer_pairs = _numeric_pairs(answer)
        contradictions = set()
        for unit in {u for _, u in gold_pairs}:
            gold_values = {n for n, u in gold_pairs if u == unit}
            answer_values = {n for n, u in answer_pairs if u == unit}
            if answer_values and not (answer_values & gold_values):
                contradictions.update("{}{}".format(n, unit) for n in sorted(answer_values))
        if contradictions:
            score -= min(6.0, 3.0 * len(contradictions))
            notes.append("정답과 다른 수치: " + ", ".join(sorted(contradictions)[:4]))

        # (2) 명시적 결론 뒤집힘 — 정답이 '아니오'인데 답변이 '예'라고 단언한 경우만
        if _has_any(" ".join(key_facts), _EXPLICIT_NO) and _has_any(answer, _EXPLICIT_YES):
            score -= 4.0
            notes.append("정답은 '아니오'인데 '예'로 답함")
        elif _has_any(" ".join(key_facts), _EXPLICIT_YES) and _has_any(answer, _EXPLICIT_NO):
            score -= 4.0
            notes.append("정답은 '예'인데 '아니오'로 답함")

        # (3) 아예 딴 얘기 — 정답 토큰이 거의 안 겹치면 사실관계를 논할 수준이 아니다
        if gold_tokens:
            overlap = len(answer_tokens & gold_tokens) / len(gold_tokens)
            if overlap < 0.10:
                score = min(score, 2.0)
                notes.append("정답 내용과 겹치는 부분이 거의 없음")

        axes["accuracy"] = max(0.0, round(score, 2))
        reasons["accuracy"] = "; ".join(notes) if notes else "정답과 어긋나는 서술 없음"

    # --- grounding: 제시한 근거가 정답 조항인가 --------------------------------------
    _, rank = score_mrr(retrieved, gold_keys)
    if not has_answer:
        axes["grounding"], reasons["grounding"] = 0.0, "답변이 비어 있음"
    elif not retrieved:
        axes["grounding"], reasons["grounding"] = 0.0, "근거 조항을 제시하지 않음"
    elif rank == 1:
        axes["grounding"], reasons["grounding"] = 10.0, "1순위 근거가 정답 조항"
    elif rank == 2:
        axes["grounding"], reasons["grounding"] = 8.0, "정답 조항이 2순위"
    elif rank == 3:
        axes["grounding"], reasons["grounding"] = 7.0, "정답 조항이 3순위"
    elif rank >= 4:
        axes["grounding"], reasons["grounding"] = 6.0, "정답 조항이 {}순위".format(rank)
    else:
        axes["grounding"], reasons["grounding"] = 2.0, "제시한 근거 중 정답 조항이 없음"

    # --- completeness: 키팩트를 빠짐없이 담았는가 -----------------------------------
    # 팀을 가르는 축. 팩트 단위로 반영 여부를 세고, 비율을 완만한 곡선으로 점수화한다.
    if not key_facts:
        axes["completeness"], reasons["completeness"] = 10.0, "골드셋에 키팩트가 없어 만점 처리"
    else:
        _, covered, _ = score_keyfact_recall(answer, key_facts)
        fraction = sum(covered) / len(covered)
        axes["completeness"] = round(10.0 * (fraction ** _COMPLETENESS_EXPONENT), 2)
        missing_idx = [i + 1 for i, ok in enumerate(covered) if not ok]
        reasons["completeness"] = "키팩트 {}개 중 {}개 반영{}".format(
            len(key_facts), sum(covered),
            (", 누락 " + str(missing_idx)) if missing_idx else "")

    # --- clarity: 읽히는 답변인가 ----------------------------------------------------
    # 공식 심판은 30건 중 2건만 깎았다. 어설픈 길이 페널티는 상관을 떨어뜨렸으므로
    # 눈에 띄게 이상한 경우만 깎는다.
    if not has_answer:
        axes["clarity"], reasons["clarity"] = 0.0, "답변이 비어 있음"
    else:
        score, notes = 10.0, []
        gold_len = len(gold_tokens) or 1
        ratio = len(tokenize(answer)) / gold_len
        if ratio > 4.0:                      # 정답의 4배가 넘는 장광설
            score -= min(2.0, (ratio - 4.0) * 0.5)
            notes.append("정답 대비 길이 {:.1f}배".format(ratio))
        counts = Counter(tokenize(answer))
        repeated = sum(c - 1 for t, c in counts.items() if c > 3 and len(t) > 1)
        if repeated > 5:                     # 같은 말 반복
            score -= min(1.5, (repeated - 5) * 0.2)
            notes.append("중복 어절 {}회".format(repeated))
        axes["clarity"] = max(0.0, round(score, 2))
        reasons["clarity"] = "; ".join(notes) if notes else "군더더기 없음"

    return axes, reasons


def judge_total_0_100(axes):
    """4축 점수를 공식 가중치로 합쳐 0~100으로 만든다."""
    return 10.0 * sum(JUDGE_AXIS_WEIGHTS[k] * float(axes.get(k, 0.0)) for k in JUDGE_AXIS_WEIGHTS)

## 5b. 판정 4축 — Gemini 3.5 Flash 실채점

**여기가 실제 제출에 쓰는 채점 로직이다.** `make_gemini_judge(api_key)` 가
`evaluate_team(..., judge_fn=...)` 에 꽂을 수 있는 채점 함수를 만들어 준다.

- **모델**: `gemini-3.5-flash` 고정. SDK는 `google-genai`(`pip install -U google-genai`).
  `generate_content`는 문서상 "레거시"지만 전면 지원되고 구조화 출력이 안정적으로
  문서화돼 있어 이 경로를 택했다. 신설 Interactions API는 2026년 6월 GA라 사례가
  아직 적어 제외했다.
- **문항 하나당 API 호출 1회** — 4축을 한 번에 JSON으로 받는다(축마다 나눠 부르지 않음).
- **temperature=0** — 완벽한 결정론은 아니지만 "같은 기준을 같은 코드로 5팀에 적용"
  요구에 맞춰 변동을 최소화한다.
- **구조화 출력**: `response_mime_type="application/json"` + JSON 스키마로 형식을
  강제한다. 점수 범위(0~10)는 스키마에 넣지 않았다 — Gemini의 스키마 서브셋이
  `minimum`/`maximum`을 항상 지원한다는 보장이 없어서다. 대신 프롬프트로 지시하고
  `_clamp_axis_score()` 로 클라이언트에서 한 번 더 강제한다.
- **재시도·폴백**: 네트워크 오류·429·JSON 파싱 실패가 `max_retries`(기본 3)번 반복되면
  **그 문항 하나만** 5절의 규칙 기반 심판으로 대체한다. 한 문항의 일시적 실패로
  나머지 수십 문항, 다른 팀의 채점까지 멈추지 않게 하기 위함이다.
- **속도 제한**: 성공 호출마다 `request_interval_s`(기본 4.5초) 만큼 쉰다 — 무료
  등급의 분당 요청 수 제한을 넘기지 않기 위한 보수적 기본값이다. 유료 등급이거나
  429가 안 뜨면 줄여도 된다.
- **stats**: `{"calls", "gemini_ok", "fallback"}` 을 실시간으로 채운다. 실행이 끝난 뒤
  `fallback` 이 크면 API 키·쿼터를 점검하고 다시 돌려야 한다 — 폴백이 많이 섞인
  결과는 "Gemini로 처리"라는 요구 조건을 만족하지 못한다.

In [ ]:
# =====================================================================================
# 5b. 판정 4축의 실제 채점 — Gemini 3.5 Flash
# =====================================================================================
# 운영진 지침: "Gemini 3.5 Flash로 공개 10문항과 비공개 30문항을 같은 코드로 처리하고,
# 익명 5팀 모두에 같은 기준을 적용해야 합니다." judge_rule_based는 API 키 없이 돌아가는
# 자체 검증·폴백용이고, 실제 제출 채점은 여기서 만드는 judge_fn을 써야 한다.
#
# SDK는 google-genai(`pip install -U google-genai`)를 쓴다. generateContent는 문서상
# "레거시"로 표시돼 있지만 여전히 전면 지원되고 구조화 출력(response_schema)이 안정적으로
# 문서화돼 있어 이 경로를 택했다 — 신설 Interactions API는 아직 사례가 적어 제외.

_JUDGE_MODEL_DEFAULT = "gemini-3.5-flash"

# score에 minimum/maximum을 넣지 않는다 — Gemini의 스키마 서브셋이 매 버전 그 필드를
# 지원한다는 보장이 없다. 범위는 프롬프트 지시 + _clamp_axis_score() 이중으로 강제한다.
_JUDGE_RESPONSE_SCHEMA = {
    "type": "object",
    "properties": {
        axis: {
            "type": "object",
            "properties": {
                "score": {"type": "integer"},
                "reason": {"type": "string"},
            },
            "required": ["score", "reason"],
        }
        for axis in JUDGE_AXIS_WEIGHTS
    },
    "required": list(JUDGE_AXIS_WEIGHTS),
}

_JUDGE_SYSTEM_PROMPT = """당신은 카카오 서비스 약관에 대한 RAG 답변을 채점하는 심사위원입니다.
아래 네 기준으로 참가자 답변을 각 0~10 정수로 채점하세요.

- accuracy(정확성): 답변에 사실과 다르거나 정답과 모순되는 내용이 있는가
- grounding(근거성): 참가자가 제시한 근거 조항이 실제 정답 조항과 일치하는가
- completeness(완결성): 정답 핵심 내용을 빠짐없이 담았는가
- clarity(명료성): 군더더기 없이 명확하고 읽기 쉬운가

각 기준마다 0~10 정수 점수와 한국어로 된 한두 문장의 근거를 반환하세요.
정답에 없는 내용을 답변이 지어냈다면 accuracy를 낮추고, 정답 조항과 다른 근거를
제시했다면 grounding을 낮추세요. 채점 기준은 모든 답변에 동일하게 적용하세요."""


def _format_retrieved_for_prompt(retrieved):
    if not retrieved:
        return "(제시한 근거 없음)"
    return "; ".join("{} {}".format(doc, article) for doc, article in retrieved)


def build_judge_prompt(question, answer, key_facts, citations, retrieved):
    """Gemini에게 보낼 채점 프롬프트 본문(시스템 지시 제외)을 만든다."""
    gold_citation_text = "; ".join(citations) if citations else "(표기 없음)"
    key_fact_text = "\n".join("- {}".format(f) for f in key_facts) if key_facts else "(없음)"
    return (
        "[질문]\n{question}\n\n"
        "[정답 근거 조항]\n{citation}\n\n"
        "[정답 핵심 내용]\n{facts}\n\n"
        "[참가자가 제시한 근거]\n{retrieved}\n\n"
        "[참가자 답변]\n{answer}\n"
    ).format(
        question=question or "(질문 없음)",
        citation=gold_citation_text,
        facts=key_fact_text,
        retrieved=_format_retrieved_for_prompt(retrieved),
        answer=answer.strip() if answer and answer.strip() else "(답변 없음)",
    )


def _clamp_axis_score(value):
    try:
        return max(0.0, min(10.0, float(value)))
    except (TypeError, ValueError):
        return 0.0


def _parse_judge_response(raw_text):
    """Gemini 응답 텍스트를 (axes, reasons)로 파싱한다. 형식이 안 맞으면 None."""
    text = (raw_text or "").strip()
    if text.startswith("```"):                              # JSON 모드에서도 가끔 코드펜스가 붙는다
        text = re.sub(r"^```[a-zA-Z]*\n?", "", text)
        text = re.sub(r"```$", "", text).strip()
    try:
        data = json.loads(text)
    except (json.JSONDecodeError, TypeError):
        return None
    if not isinstance(data, dict):
        return None

    axes, reasons = {}, {}
    for axis in JUDGE_AXIS_WEIGHTS:
        entry = data.get(axis)
        if not isinstance(entry, dict) or "score" not in entry:
            return None
        axes[axis] = _clamp_axis_score(entry.get("score"))
        reason = entry.get("reason")
        reasons[axis] = reason.strip() if isinstance(reason, str) and reason.strip() else "(사유 없음)"
    return axes, reasons


def make_gemini_judge(api_key, model=_JUDGE_MODEL_DEFAULT, max_retries=3,
                       retry_wait_s=8.0, request_interval_s=4.5, stats=None):
    """Gemini 3.5 Flash로 채점하는 judge_fn을 만든다.

    `evaluate_team(..., judge_fn=make_gemini_judge(키))` 형태로 꽂아 쓴다.

    한 문항에서 네트워크 오류·429·JSON 파싱 실패가 max_retries번 반복되면 그 문항만
    judge_rule_based로 대체하고 사유에 남긴다. 한 문항의 일시적 실패로 나머지 수십
    문항, 다른 팀의 채점까지 중단되는 것을 막기 위함이다. `stats`(dict)를 넘기면
    {"calls", "gemini_ok", "fallback"} 진행 상황이 실시간으로 쌓인다 — 채점이 끝난
    뒤 fallback이 크면 API 키·쿼터를 점검하고 다시 돌려야 한다는 신호다.
    """
    from google import genai
    from google.genai import errors as genai_errors

    client = genai.Client(api_key=api_key)
    if stats is None:
        stats = {}
    stats.setdefault("calls", 0)
    stats.setdefault("gemini_ok", 0)
    stats.setdefault("fallback", 0)

    def judge_fn(question, answer, key_facts, citations, retrieved, gold_keys):
        stats["calls"] += 1
        prompt = build_judge_prompt(question, answer, key_facts, citations, retrieved)
        last_error = "알 수 없는 오류"

        for attempt in range(max_retries):
            if attempt > 0:
                time.sleep(retry_wait_s * attempt)
            try:
                response = client.models.generate_content(
                    model=model,
                    contents=prompt,
                    config={
                        "system_instruction": _JUDGE_SYSTEM_PROMPT,
                        "response_mime_type": "application/json",
                        "response_schema": _JUDGE_RESPONSE_SCHEMA,
                        "temperature": 0,
                    },
                )
                parsed = _parse_judge_response(getattr(response, "text", None))
                if parsed is not None:
                    stats["gemini_ok"] += 1
                    time.sleep(request_interval_s)          # 무료 등급 분당 호출 수 방어
                    return parsed
                last_error = "응답을 정해진 JSON 형식으로 해석할 수 없음"
            except genai_errors.APIError as exc:
                last_error = "{} (code={})".format(exc, getattr(exc, "code", "?"))
            except Exception as exc:
                last_error = str(exc)

        stats["fallback"] += 1
        axes, reasons = judge_rule_based(question, answer, key_facts, citations, retrieved, gold_keys)
        reasons = {axis: "[Gemini {}회 실패, 규칙기반 대체: {}] {}".format(max_retries, last_error, text)
                   for axis, text in reasons.items()}
        return axes, reasons

    return judge_fn


def _resolve_qid_mapping(goldset, answers):
    """답변 qid가 골드셋 id와 하나도 안 겹치는데 개수는 같으면, 제출 순서를 골드셋
    순서에 그대로 대응시킨다(위치 기반 폴백).

    실제로 있었던 상황: 팀 결과기가 낸 정상 qid(P01..)를, 운영진이 답변 파일을
    익명화·재배포하는 과정에서 BLIND01..처럼 통째로 다시 붙이는 경우. 이때 문자열
    매칭만 하면 내용은 멀쩡한데 전부 "미응답 0점"으로 잘못 채점된다.

    조금이라도 겹치면(부분 일치) 폴백하지 않는다 — 애매하게 일부만 다른 경우는
    정말 일부 문항이 빠진 것일 수 있고, 그럴 때 위치로 임의 매칭하면 엉뚱한 답을
    엉뚱한 문항에 채점하는 더 나쁜 실패로 이어진다. 완전히 안 겹치고 개수까지
    맞을 때만 순서로 넘어간다. answers의 순서는 load_answers가 만든 딕셔너리라
    파이썬 3.7+ 삽입 순서 보장으로 원본 답변 파일 순서와 같다.

    반환: (실제로 조회에 쓸 answers 딕셔너리, 폴백을 썼다면 사유 문자열 또는 None)
    """
    gold_ids = [q["id"] for q in goldset.questions]
    answer_qids = list(answers.keys())
    if not answer_qids or (set(gold_ids) & set(answer_qids)):
        return answers, None
    if len(answer_qids) != len(gold_ids):
        return answers, None

    remapped = {gold_ids[i]: answers[answer_qids[i]] for i in range(len(gold_ids))}
    note = ("qid가 골드셋 id와 전혀 겹치지 않아(예: {!r} vs {!r}) 제출 순서를 골드셋 "
            "순서에 그대로 대응시켰습니다({}개 전부).").format(answer_qids[0], gold_ids[0], len(gold_ids))
    return remapped, note

## 6. 한 팀 채점

**골드셋의 문항을 기준으로 돈다.** 답변 파일에 없는 문항은 0점 + 사유 기록.
답변 파일에만 있고 골드셋에 없는 문항은 그냥 무시한다. 문항 수·id는 전부
골드셋에서 오므로 10문항이든 30문항이든 코드는 그대로다.

**`qid`가 골드셋 `id`와 전혀 다른 라벨을 쓰는 경우** — 예를 들어 골드셋은
`P01`인데 답변 파일은 `BLIND01`처럼 완전히 다른 체계를 쓰는 경우 —
`_resolve_qid_mapping()` 이 **제출 순서를 골드셋 순서에 그대로 대응**시킨다.
실제로 이런 사례가 있었다: 팀 결과기가 낸 정상 qid를 운영진이 답변 파일을
재배포하며 통째로 다시 붙이는 경우, 문자열 매칭만 하면 내용은 멀쩡한 답변이
전부 "미응답 0점"으로 잘못 채점된다.

**개수가 같고 하나도 안 겹칠 때만** 이 폴백이 켜진다. 하나라도 겹치면(부분
일치) 켜지지 않는다 — 그런 애매한 경우는 정말 일부 문항이 빠진 것일 수 있고,
그때 위치로 임의 매칭하면 엉뚱한 답을 엉뚱한 문항에 채점하는 더 나쁜 실패로
이어지기 때문이다. 폴백이 켜지면 `report["qid_mapping_note"]` 에 사유가 남고,
`evaluate_submissions` 를 통하면 `details[blind_id]["problems"]` 로도 보인다.

In [ ]:
# =====================================================================================
# 6. 한 팀 채점 — 문항별 점수 → 총점
# =====================================================================================
def evaluate_team(goldset, answers, judge_fn=judge_rule_based):
    """골드셋 한 벌과 팀 답변 하나를 맞대어 채점한다.

    골드셋의 문항을 기준으로 돈다. 답변에 없는 문항은 0점 처리하고 사유를 남긴다.
    문항 수·id는 전부 goldset에서 온다(코드에 고정된 값 없음).
    """
    answers, qid_note = _resolve_qid_mapping(goldset, answers)

    per_question = []
    for question in goldset.questions:
        qid = question["id"]
        entry = answers.get(qid)

        if entry is None:
            per_question.append({
                "qid": qid, "answered": False,
                "mrr": 0.0, "rank": 0,
                "keyfact_f1": 0.0, "keyfact_precision": 0.0, "keyfact_recall": 0.0,
                "judge_axes": {k: 0.0 for k in JUDGE_AXIS_WEIGHTS},
                "judge_total": 0.0,
                "note": "답변 파일에 이 문항이 없음",
            })
            continue

        answer, retrieved = entry["answer"], entry["retrieved"]
        mrr, rank = score_mrr(retrieved, question["gold_keys"])
        f1, precision, recall_tok = score_keyfact_f1(answer, question["key_facts"])
        axes, reasons = judge_fn(
            question=question["question"], answer=answer,
            key_facts=question["key_facts"], citations=question["citations"],
            retrieved=retrieved, gold_keys=question["gold_keys"],
        )
        per_question.append({
            "qid": qid, "answered": True,
            "mrr": round(mrr, 6), "rank": rank,
            "keyfact_f1": round(f1, 6),
            "keyfact_precision": round(precision, 6),
            "keyfact_recall": round(recall_tok, 6),
            "judge_axes": axes, "judge_reasons": reasons,
            "judge_total": round(judge_total_0_100(axes), 4),
            "note": "",
        })

    n = len(per_question) or 1
    mrr_mean = sum(p["mrr"] for p in per_question) / n
    f1_mean = sum(p["keyfact_f1"] for p in per_question) / n
    judge_mean = sum(p["judge_total"] for p in per_question) / n

    total = (WEIGHTS["mrr"] * mrr_mean * 100
             + WEIGHTS["keyfact"] * f1_mean * 100
             + WEIGHTS["judge"] * judge_mean)

    return {
        "objective": {"mrr": round(mrr_mean, 6), "keyfact_f1": round(f1_mean, 6)},
        "judge": {"score_0_100": round(judge_mean, 4)},
        "total_0_100": round(total, 4),
        "n_questions": len(per_question),
        "n_answered": sum(1 for p in per_question if p["answered"]),
        "per_question": per_question,
        "weights": dict(WEIGHTS),
        "qid_mapping_note": qid_note,
    }

## 7. 교차평가 → `eval_8.json`

`blind_id` 는 이 순서로 정한다: 파일 안 `blind_id` 필드 → 파일명의 `BLIND\d+`(숫자는
항상 `BLIND01`처럼 2자리로 맞춤) → 파일명 나머지 → `team` 필드 → 순번.

**`status` 는 항상 `"completed"` 다.** 운영진이 배포한 "교차 평가 결과 파일 검사기"의
규칙 때문이다 — *"일부 문항 실패·평가 전체 실패·후보 중복·누락이 하나라도 있으면
그 평가 결과 파일 전체가 순위 산정에서 제외됩니다. 실패한 후보에게 0점을 주는 방식은
사용하지 않습니다."* 즉 `partial`/`failed` 는 그 후보만 감점되는 게 아니라 **내 제출
파일 전체를 무효화**시킨다. 그래서 후보 파일이 아예 안 읽혀도 빈 답변(`{}`)으로
채점을 강행해 실제 점수(대개 0점에 가까움)를 매기고 `completed` 로 마감한다 —
`load_answers`/`evaluate_team` 이 이미 빈 입력을 정상적인 0점 만점 채점으로 처리하도록
돼 있어(§2, §6) 가능하다. 채점 중 무슨 문제가 있었는지는 제출 파일이 아니라
`details[blind_id]["problems"]` 에만 남는다(진단용).

`check_blind_id_coverage()` 는 제출 직전에 `BLIND01`~`BLIND05` 가 정확히 한 번씩만
있는지 확인한다 — 운영진 검사기가 "후보 중복"·"누락"으로 잡는 것과 같은 조건이다.

In [ ]:
# =====================================================================================
# 7. 교차평가 — 여러 팀 파일 → eval_<우리팀>.json
# =====================================================================================
# 운영진 검사기(교차 평가 결과 파일 검사기)의 규칙: results에는 BLIND01~BLIND05가
# 각각 정확히 한 번씩 있어야 하고, 다섯 결과의 status가 모두 completed여야 제출할 수
# 있다. "일부 문항 실패·평가 전체 실패·후보 중복·누락이 하나라도 있으면 그 평가 결과
# 파일 전체가 순위 산정에서 제외됩니다. 실패한 후보에게 0점을 주는 방식은 사용하지
# 않습니다." — 즉 후보 파일이 아무리 깨져 있어도 failed/partial로 도망치면 안 되고,
# 항상 실제 점수(깨졌으면 낮은 점수)를 매겨 completed로 마감해야 한다. 그래서
# evaluate_submissions는 파일이 아예 안 읽혀도 빈 답변({})으로 채점을 강행한다 —
# load_answers/evaluate_team이 이미 빈 입력을 0점 만점 정상 채점으로 처리하도록
# 설계돼 있어서 가능하다(§2, §6).
BLIND_IDS = tuple("BLIND{:02d}".format(i) for i in range(1, 6))

_BLIND_RE = re.compile(r"BLIND\s*[-_]?\s*(\d+)", re.IGNORECASE)


def resolve_blind_id(info, path, fallback_index):
    """블라인드 식별자를 정한다. 파일 안 blind_id > 파일명(basename) > team > 순번.

    숫자가 발견되면 항상 BLIND01처럼 2자리로 맞춘다 — 파일명이 BLIND1.json이든
    blind_01.json이든 운영진 검사기가 요구하는 BLIND01 형식으로 귀결시키기 위함이다.

    파일명(basename)만 본다 — 전체 경로를 보면 상위 폴더 이름에 숫자가 섞였을 때
    엉뚱한 값으로 오매칭될 수 있다(예: "blind5/answers_BLIND01.json" 같은 폴더
    구조에서 폴더명의 5를 먼저 집어 모든 파일이 BLIND05로 뭉개지는 사고가 실제로
    있었다).
    """
    basename = str(path).replace("\\", "/").rsplit("/", 1)[-1]
    for candidate in (info.get("blind_id"), basename):
        if not candidate:
            continue
        match = _BLIND_RE.search(str(candidate))
        if match:
            return "BLIND{:02d}".format(int(match.group(1)))
    stem = str(path).replace("\\", "/").rsplit("/", 1)[-1]
    stem = re.sub(r"\.json$", "", stem, flags=re.IGNORECASE)
    stem = re.sub(r"^answers[_-]?(public|blind|private)?[_-]?", "", stem, flags=re.IGNORECASE)
    if stem.strip():
        return stem.strip()
    if info.get("team"):
        return info["team"]
    return "ENTRY{:02d}".format(fallback_index)


def evaluate_submissions(goldset, submissions, judge_fn=judge_rule_based):
    """제출물 여러 개를 채점한다.

    submissions: [경로, ...] 또는 [(blind_id, 경로_또는_payload), ...]
    반환: {"results": [{blind_id, total, status}, ...], "details": {...}}

    status는 항상 "completed"다. 후보 파일을 전혀 읽을 수 없어도 빈 답변으로
    채점을 강행해 실제 점수를 매긴다 — "실패한 후보에게 0점을 주는 방식은
    사용하지 않는다"는 운영진 지침대로, failed/partial로 파일 전체를 무효화시킬
    위험을 피하기 위함이다. 문제가 있었다면 그 사유는 결과 파일이 아니라
    details[blind_id]["problems"]에만 남는다(진단용, 제출 파일에는 안 들어감).
    """
    results, details = [], {}

    for index, item in enumerate(submissions, 1):
        if isinstance(item, (tuple, list)) and len(item) == 2:
            given_id, source = item
        else:
            given_id, source = None, item

        answers, info = load_answers(source)
        blind_id = given_id or resolve_blind_id(info, source, index)

        try:
            report = evaluate_team(goldset, answers, judge_fn=judge_fn)
        except Exception as exc:
            info["problems"] = info["problems"] + ["채점 중 오류(0점 처리): {}".format(exc)]
            report = evaluate_team(goldset, {}, judge_fn=judge_fn)

        problems = info["problems"]
        if report.get("qid_mapping_note"):
            problems = problems + [report["qid_mapping_note"]]

        results.append({
            "blind_id": blind_id,
            "total": round(report["total_0_100"], 4),
            "status": "completed",
        })
        report["problems"] = problems
        report["source"] = info["source"]
        details[blind_id] = report

    results.sort(key=lambda r: str(r["blind_id"]))
    return {"results": results, "details": details}


def check_blind_id_coverage(results):
    """제출 직전 확인: BLIND01~BLIND05가 정확히 한 번씩만 있는가.

    운영진 검사기가 "후보 중복"·"누락"으로 잡는 것과 같은 조건이다. 문제 없으면 빈
    리스트, 있으면 사람이 읽을 문구 목록을 돌려준다.
    """
    issues = []
    seen_order = [str(r.get("blind_id")) for r in results]
    counts = Counter(seen_order)
    duplicates = sorted(bid for bid, n in counts.items() if n > 1)
    if duplicates:
        issues.append("blind_id가 중복되었습니다: {}".format(", ".join(duplicates)))
    unexpected = sorted(set(seen_order) - set(BLIND_IDS))
    if unexpected:
        issues.append("BLIND01~BLIND05가 아닌 값이 있습니다: {}".format(", ".join(unexpected)))
    missing = sorted(set(BLIND_IDS) - set(seen_order))
    if missing:
        issues.append("누락된 익명 후보가 있습니다: {}".format(", ".join(missing)))
    return issues


def write_eval_file(payload, team, out_dir="."):
    """eval_<팀>.json 을 규정 형식(results[{blind_id,total,status}])으로 저장한다."""
    path = "{}/eval_{}.json".format(out_dir.rstrip("/"), team)
    with open(path, "w", encoding="utf-8") as fp:
        json.dump({"results": payload["results"]}, fp, ensure_ascii=False, indent=2)
        fp.write("\n")
    return path

## 7b. 제출 파일 형식 검사 — 운영진 검사기와 동일 기준

운영진이 배포한 "[학생용] 교차 평가 결과 파일 검사기"를 그대로 옮겼다. 재구현하지
않고 원본을 옮긴 이유는, 다시 구현하는 과정에서 조건 하나라도 다르게 해석하면
**"내 검사는 통과했는데 실제 제출은 반려"** 되는 상황이 생기기 때문이다 —
`schema_version`/`rank` 금지, `BLIND01~05` 정확히 5개, `status` 전부 `completed`,
`total` 범위·null 규칙까지 운영진 코드와 1:1로 같다.

10절에서 `eval_8.json` 을 만들자마자 이 검사를 바로 돌려 `[제출 가능]` 을 확인한다.
운영진이 새 검사기를 배포하면 이 셀 내용을 그걸로 통째로 교체하면 된다.

In [ ]:
# =====================================================================================
# 7b. 제출 파일 형식 검사 — 운영진 "교차 평가 결과 파일 검사기"와 동일 기준
# =====================================================================================
# 운영진이 배포한 검사기 코드를 그대로 옮겼다(문항 검사기와 같은 방식). 직접 다시
# 구현하지 않고 원본을 옮긴 이유는, 재구현 과정에서 조건 하나라도 다르게 해석하면
# "내 검사는 통과했는데 실제 제출은 반려"되는 상황이 생기기 때문이다. 새 검사기가
# 배포되면 이 구획을 그걸로 통째로 교체하면 된다.

_EVAL_ROOT_KEYS = {"results"}
_EVAL_RESULT_KEYS = {"blind_id", "total", "status"}
_EVAL_STATUSES = {"completed", "partial", "failed"}
EVAL_FILENAME_RE = re.compile(r"^eval_([1-9]|1[0-7])\.json$")


def _eval_number(value):
    if isinstance(value, bool) or not isinstance(value, (int, float)):
        return None
    number = float(value)
    return number if math.isfinite(number) else None


def validate_eval_document(document):
    """운영진 취합기의 최소 계약과 같은 기준으로 검사하고 문제 문구 목록을 반환한다."""
    issues = []
    if not isinstance(document, dict):
        return ["최상위 값은 JSON 객체여야 합니다."]

    if "schema_version" in document:
        issues.append("schema_version은 학생 제출 파일에 저장하지 않습니다.")
    if "rank" in document:
        issues.append("rank는 저장하지 않습니다. 운영진이 반올림 전 total에서 계산합니다.")
    extra_root_keys = sorted(set(document) - _EVAL_ROOT_KEYS - {"schema_version", "rank"})
    if extra_root_keys:
        issues.append("최상위에는 results만 저장합니다. 허용되지 않은 항목: " + ", ".join(extra_root_keys))

    raw_results = document.get("results")
    if not isinstance(raw_results, list):
        issues.append("results는 배열이어야 합니다.")
        return issues

    seen = set()
    for index, item in enumerate(raw_results):
        at = "results[{}]".format(index)
        if not isinstance(item, dict):
            issues.append("{}는 JSON 객체여야 합니다.".format(at))
            continue
        if "rank" in item:
            issues.append("{}.rank는 저장하지 않습니다.".format(at))
        extra_result_keys = sorted(set(item) - _EVAL_RESULT_KEYS - {"rank"})
        if extra_result_keys:
            issues.append("{}에는 blind_id, total, status만 저장합니다. 허용되지 않은 항목: {}"
                          .format(at, ", ".join(extra_result_keys)))

        blind_id = str(item.get("blind_id") or "").strip()
        if blind_id not in BLIND_IDS:
            issues.append("{}.blind_id는 BLIND01~BLIND05 중 하나여야 합니다: {!r}".format(at, blind_id))
            continue
        if blind_id in seen:
            issues.append("{}가 두 번 이상 들어 있습니다.".format(blind_id))
            continue
        seen.add(blind_id)

        status = str(item.get("status") or "").strip()
        if status not in _EVAL_STATUSES:
            issues.append("{}.status는 completed·partial·failed 중 하나여야 합니다: {!r}".format(blind_id, status))
            continue

        raw_total = item.get("total")
        total = _eval_number(raw_total)
        if status == "failed":
            if raw_total is not None:
                issues.append("{}가 failed이면 total은 null이어야 합니다.".format(blind_id))
        elif total is None or not 0 <= total <= 100:
            issues.append("{}.total은 0~100 숫자여야 합니다.".format(blind_id))

        if status != "completed":
            issues.append("{} 상태가 {}라 평가 결과 파일 전체가 순위 산정에서 제외됩니다.".format(blind_id, status))

    missing = sorted(set(BLIND_IDS) - seen)
    if missing:
        issues.append("익명 후보가 누락됐습니다: {}".format(", ".join(missing)))
    if len(raw_results) != len(BLIND_IDS):
        issues.append("results는 정확히 5개여야 합니다: 현재 {}개".format(len(raw_results)))
    return issues


def check_eval_file(path):
    """eval_<팀>.json 파일을 읽어 (제출 가능 여부, 문제 문구 목록)을 반환한다."""
    name = str(path).replace("\\", "/").rsplit("/", 1)[-1]
    if not EVAL_FILENAME_RE.fullmatch(name):
        return False, ["파일명은 eval_1.json부터 eval_17.json까지의 형식이어야 합니다. 현재 파일명: {}".format(name)]
    try:
        with open(path, encoding="utf-8") as fp:
            document = json.load(fp)
    except (OSError, json.JSONDecodeError) as exc:
        return False, ["JSON 파일을 읽지 못했습니다: {}".format(exc)]
    issues = validate_eval_document(document)
    return not issues, issues

## 8. 자체 검증 (API 키 불필요)

채점을 돌리기 전에 이 셀로 평가기 배선이 제대로 도는지 확인한다.
가짜 골드셋(id `B001`~`B030`)과 일부러 깨뜨린 답변 파일들을 만들어
만점·영점·깨진 파일 처리, 그리고 **다섯 후보가 파일 상태와 무관하게 항상
`completed`로 마감되는지**(7절 참고)까지 확인한다. 마지막엔 운영진 검사기 기준
(`validate_eval_document`, 7b절)으로도 문제 0건인지 직접 검사한다.

**id 체계와 문항 수가 공개셋과 다른 골드셋으로 도는 것**을 여기서 확인하므로,
비공개 30문항 골드셋을 받아도 그대로 쓸 수 있다는 근거가 된다.

이 셀은 5절의 규칙 기반 심판으로 동작한다 — Gemini API 키가 아직 없어도 실행할 수
있게 하기 위해서다. 점수 자체(축 보정)가 아니라 **채점 파이프라인의 배선**을
확인하는 것이 목적이며, 9절에서 Gemini 연결도 별도로 스모크 테스트한다.

In [ ]:

def _self_test():
    """가짜 입력으로 평가기 동작을 확인한다. 실패하면 목록을 찍는다."""
    ok, bad = [], []

    def check(name, cond, detail=""):
        if cond:
            ok.append(name)
        else:
            bad.append(name + ("  <- " + detail if detail else ""))

    # --- 정규화 ---
    check("조번호 '제10조' -> 10", parse_article_no("제10조") == 10)
    check("조번호 True는 무효", parse_article_no(True) is None)
    check("문서명 공백 무시", normalize_doc_name("카카오 통합 약관") == normalize_doc_name("카카오통합약관"))
    check("통합서비스약관/통합약관 구분",
          normalize_doc_name("카카오 통합서비스약관") != normalize_doc_name("카카오 통합 약관"))
    check("가운뎃점 분리", tokenize("수집·이용·제공") == ["수집", "이용", "제공"])

    # --- 지표 ---
    gk = [("카카오계정약관", 10), ("카카오통합약관", 13)]
    check("MRR 1순위", score_mrr([("카카오계정 약관", 10)], gk) == (1.0, 1))
    check("MRR 2순위", score_mrr([("카카오 통합 약관", 99), ("카카오계정 약관", 10)], gk)[0] == 0.5)
    check("MRR 복수 정답 중 하나", score_mrr([("카카오 통합 약관", 13)], gk) == (1.0, 1))
    check("MRR 실패", score_mrr([("카카오계정 약관", 99)], gk) == (0.0, 0))
    facts = ["담당자 1인만 이용할 수 있습니다.", "공유하는 것은 금지됩니다."]
    check("F1 정답 그대로면 1.0", abs(score_keyfact_f1(" ".join(facts), facts)[0] - 1.0) < 1e-9)
    check("F1 무관한 답변은 0", score_keyfact_f1("오늘 날씨가 좋습니다", facts)[0] == 0.0)
    check("F1 군더더기로 떨어짐",
          score_keyfact_f1(" ".join(facts) + " 관련 없는 말 " * 30, facts)[0]
          < score_keyfact_f1(" ".join(facts), facts)[0])

    # --- 공개셋과 다른 id 체계 / 다른 문항 수 ---
    fake = {"questions": [
        {"id": "B{:03d}".format(i), "question": "질문 {}".format(i),
         "gold_articles": [{"doc": "카카오계정 약관", "article": i, "citation": "제{}조".format(i)}],
         "key_facts": ["문항 {}의 핵심 사실 하나입니다.".format(i),
                       "문항 {}의 핵심 사실 둘입니다.".format(i)],
         "ptype": "basic", "difficulty": "hard", "운영진메모": "무시돼야 함"}
        for i in range(1, 31)]}
    gs = load_goldset(fake)
    check("골드셋 30문항 로드", len(gs) == 30)
    check("id 체계가 달라도 됨", gs.qids[0] == "B001" and gs.qids[-1] == "B030")
    check("추가 필드는 안 읽음",
          set(gs.questions[0]) == {"id", "question", "gold_keys", "citations", "key_facts"})

    def answers_of(items):
        return load_answers({"team": "T", "answers": items})[0]

    perfect = answers_of([{"qid": q["id"], "retrieved": [["카카오계정 약관", int(q["id"][1:])]],
                           "answer": " ".join(q["key_facts"])} for q in fake["questions"]])
    check("정답 그대로면 총점 100", abs(evaluate_team(gs, perfect)["total_0_100"] - 100.0) < 1e-6)

    empty = answers_of([{"qid": q["id"], "retrieved": [], "answer": ""} for q in fake["questions"]])
    check("전부 빈 답변이면 0점", evaluate_team(gs, empty)["total_0_100"] == 0.0)

    half = answers_of([{"qid": q["id"], "retrieved": [["카카오계정 약관", int(q["id"][1:])]],
                        "answer": " ".join(q["key_facts"])} for q in fake["questions"][:15]])
    rep_half = evaluate_team(gs, half)
    check("절반만 답하면 총점도 절반쯤", 45 < rep_half["total_0_100"] < 55)
    check("없는 문항은 0점 + 사유", any(p["note"] for p in rep_half["per_question"] if not p["answered"]))

    wrong = answers_of([{"qid": q["id"], "retrieved": [["카카오 위치정보 이용약관", 99]],
                         "answer": " ".join(q["key_facts"])} for q in fake["questions"]])
    rep_wrong = evaluate_team(gs, wrong)
    check("근거 틀리면 MRR 0", rep_wrong["objective"]["mrr"] == 0.0)
    check("근거 틀리면 grounding 감점",
          rep_wrong["per_question"][0]["judge_axes"]["grounding"] < 5)

    # --- 깨진 답변 파일 ---
    _, info = load_answers({"team": "x"})
    check("answers 없으면 fatal", info["fatal"])
    weird, winfo = load_answers({"team": "x", "answers": [
        {"qid": "B001", "retrieved": [{"doc": "카카오계정 약관", "article": "제1조"}], "answer": "정상"},
        {"qid": "B002", "retrieved": "목록아님", "answer": "그래도 채점됨"},
        {"qid": "B003", "retrieved": [["문서명만"]], "answer": "쌍이 깨짐"},
        {"qid": "", "answer": "qid 없음"},
        {"qid": "B004", "answer": None},
        "항목이 문자열",
    ]})
    check("이상한 항목이 섞여도 예외 없음", len(weird) == 4)
    check("dict 형태 retrieved 인정",
          score_mrr(weird["B001"]["retrieved"], [("카카오계정약관", 1)]) == (1.0, 1))
    check("answer가 null이면 빈 문자열", weird["B004"]["answer"] == "")

    # --- 교차평가 산출물: BLIND01~05 다섯 다 있어야 하고, status는 항상 completed ---
    # 운영진 검사기 규칙: 하나라도 completed가 아니면 제출 파일 전체가 순위 산정에서
    # 제외된다. 그래서 못 읽는 파일(BLIND02)·문항이 모자란 파일(BLIND03)도
    # failed/partial로 도망치지 않고 completed + 실제 점수(대개 낮음)로 마감해야 한다.
    out = evaluate_submissions(gs, [
        ("BLIND01", {"team": "1", "answers": [
            {"qid": q["id"], "retrieved": [["카카오계정 약관", int(q["id"][1:])]],
             "answer": " ".join(q["key_facts"])} for q in fake["questions"]]}),
        ("BLIND02", {"team": "2"}),                              # answers 필드 자체가 없음
        ("BLIND03", {"team": "3", "answers": [
            {"qid": q["id"], "retrieved": [], "answer": "x"} for q in fake["questions"][:5]]}),
        ("BLIND04", {"team": "4", "answers": []}),
        ("BLIND05", {"team": "5", "answers": [
            {"qid": q["id"], "retrieved": [["카카오계정 약관", int(q["id"][1:])]],
             "answer": " ".join(q["key_facts"])} for q in fake["questions"]]}),
    ])
    res = {r["blind_id"]: r for r in out["results"]}
    check("완주 -> completed, 100점", res["BLIND01"]["status"] == "completed" and res["BLIND01"]["total"] == 100.0)
    check("answers 필드가 아예 없어도 completed + 0점(failed 아님)",
          res["BLIND02"]["status"] == "completed" and res["BLIND02"]["total"] == 0.0)
    check("문항이 모자라도 completed(partial 아님)", res["BLIND03"]["status"] == "completed")
    check("빈 배열도 completed + 0점", res["BLIND04"]["status"] == "completed" and res["BLIND04"]["total"] == 0.0)
    check("다섯 결과 status가 전부 completed", all(r["status"] == "completed" for r in out["results"]))
    check("결과 키는 셋뿐", all(set(r) == {"blind_id", "total", "status"} for r in out["results"]))
    check("total은 0~100", all(0.0 <= r["total"] <= 100.0 for r in out["results"]))

    coverage_issues = check_blind_id_coverage(out["results"])
    check("BLIND01~05 커버리지 문제 없음", coverage_issues == [])

    validation_issues = validate_eval_document({"results": out["results"]})
    check("운영진 검사기 기준으로도 제출 가능(문제 0건)", validation_issues == [], str(validation_issues))

    # --- qid가 골드셋 id와 전혀 다른 라벨(예: BLIND01)이어도 위치로 매칭되는지 ---
    relabeled = {"team": "relabel-test", "answers": [
        {"qid": "ZZZ{:02d}".format(i + 1), "retrieved": [["카카오계정 약관", int(q["id"][1:])]],
         "answer": " ".join(q["key_facts"])} for i, q in enumerate(fake["questions"])]}
    rep_relabel = evaluate_team(gs, load_answers(relabeled)[0])
    check("qid가 골드셋 id와 완전히 달라도(개수는 같음) 위치로 매칭돼 만점",
          abs(rep_relabel["total_0_100"] - 100.0) < 1e-6, "총점 {}".format(rep_relabel["total_0_100"]))
    check("정상 qid에는 폴백 노트가 안 붙음", evaluate_team(gs, perfect)["qid_mapping_note"] is None)

    print("자체 검증: 통과 {} / 실패 {}".format(len(ok), len(bad)))
    for name in bad:
        print("  [실패] " + name)
    return not bad


_SELF_TEST_OK = _self_test()

## 9. Gemini API 키 설정 + 스모크 테스트

10절에서 골드셋·답변 파일을 올리기 전에, 여기서 API 키와 모델 호출이 실제로 되는지
문항 1개로 먼저 확인한다. 200문항 가까이(5팀 × 최대 40문항) 처리하다가 중간에
키 오류를 발견하는 것보다 훨씬 싸게 고칠 수 있다.

**키 발급**: https://aistudio.google.com 에서 Google 계정으로 로그인 → API 키 생성.
`AIza` 로 시작하는 문자열이다. **코드에 직접 적지 말 것** — 아래 셀이 입력창으로
물어보거나(화면에 표시되지 않음), Colab 보안 비밀(왼쪽 열쇠 아이콘)에 `GEMINI_API_KEY`
로 저장해 두면 자동으로 읽는다.

In [ ]:

!pip install -q -U google-genai

import getpass


def _resolve_gemini_api_key():
    try:
        from google.colab import userdata
        key = userdata.get("GEMINI_API_KEY")
        if key:
            return key
    except Exception:
        pass
    env_key = os.environ.get("GEMINI_API_KEY")
    if env_key:
        return env_key
    return getpass.getpass("Gemini API 키를 입력하세요 (화면에 표시되지 않습니다): ").strip()


GEMINI_API_KEY = _resolve_gemini_api_key()
if not GEMINI_API_KEY:
    raise RuntimeError(
        "Gemini API 키가 없습니다. https://aistudio.google.com 에서 발급받아 "
        "다시 실행하며 입력하세요.")

judge_stats = {"calls": 0, "gemini_ok": 0, "fallback": 0}
gemini_judge = make_gemini_judge(GEMINI_API_KEY, stats=judge_stats)
print("Gemini 판정 준비 완료 (모델: {})".format(_JUDGE_MODEL_DEFAULT))

_smoke_axes, _smoke_reasons = gemini_judge(
    question="사업자/단체 카카오계정은 담당자 몇 명이 이용할 수 있나요?",
    answer="담당자 1인만 이용할 수 있으며, 다른 사람과 공유하는 것은 금지됩니다.",
    key_facts=["담당자 1인만 이용할 수 있습니다.", "다른 사람에게 공유하는 것은 금지됩니다."],
    citations=["카카오계정 약관 제10조"],
    retrieved=[["카카오계정 약관", 10]],
    gold_keys=[("카카오계정약관", 10)],
)
print("\n스모크 테스트 결과 (정답에 가까운 답변이므로 대체로 고득점이 정상):")
for axis, score in _smoke_axes.items():
    print("  {:<13s} {:>4.1f}   {}".format(axis, score, _smoke_reasons[axis]))

if judge_stats["fallback"] > 0:
    raise RuntimeError(
        "Gemini 호출이 실패해 규칙 기반으로 대체됐습니다. 위 사유를 확인하고 "
        "키·쿼터·네트워크를 점검한 뒤 이 셀을 다시 실행하세요.")
print("\nGemini 연결 정상 — 10절에서 실제 채점을 진행하면 됩니다.")

## 10. 실행 — 골드셋과 답변 파일을 넣고 `eval_8.json` 만들기

**Colab**: 아래 셀을 실행하면 파일 선택 창이 뜬다. 골드셋 1개 + 채점할 답변 파일들을
한꺼번에 고르면 된다. 골드셋은 `questions` 가 들어 있는 파일로 자동 식별한다.

**로컬**: `USE_UPLOAD = False` 로 두고 `GOLD_PATH` / `ANSWER_PATHS` 에 경로를 적는다.

이 셀은 **두 가지 용도**로 그대로 쓴다. 답변 파일 개수로 자동 구분한다.

- **개발·연습(답변 파일 1개 이상, 5개가 아님)** — 결과기가 만든 자기 팀
  `answers_public_<팀>.json` 을 그대로 채점해 점수만 확인한다. BLIND01~05
  구성·제출 형식 검사는 **건너뛴다** — 아직 제출할 게 아니므로 당연하다.
- **본선 제출(답변 파일 정확히 5개)** — 익명 5팀 답변 파일을 채점해
  `eval_8.json` 을 만든다. 이때만 BLIND01~05 구성을 엄격히 검사하고, 저장 직후
  7b절 검사기로 `[제출 가능]`/`[확인 필요]` 를 바로 보여 준다.

문항마다 Gemini 호출 1회 + 속도 제한 대기가 들어가므로, 5팀 × 30문항 기준으로
**대략 10~20분** 걸린다고 보면 된다. 진행 중 문항마다 한 줄씩 진행 상황이 찍힌다.

In [ ]:

# ═══════════════════════════════════════════════════════════════
#  우리 팀 번호 — 결과 파일이 eval_<이 값>.json 으로 저장된다
TEAM = "8"
# ═══════════════════════════════════════════════════════════════

USE_UPLOAD = True          # Colab에서 파일을 골라 올릴지 여부
GOLD_PATH = ""             # USE_UPLOAD=False 일 때 쓰는 골드셋 경로
ANSWER_PATHS = []          # USE_UPLOAD=False 일 때 쓰는 답변 파일 경로 목록
OUT_DIR = "."


def _looks_like_goldset(path):
    """questions[] 를 가진 파일을 골드셋으로 본다."""
    try:
        with open(path, encoding="utf-8") as fp:
            payload = json.load(fp)
    except Exception:
        return False
    return isinstance(payload, dict) and isinstance(payload.get("questions"), list)


def collect_inputs():
    """골드셋 1개와 답변 파일 목록을 모은다."""
    if USE_UPLOAD:
        try:
            from google.colab import files
        except ImportError:
            raise RuntimeError(
                "Colab이 아닙니다. USE_UPLOAD = False 로 바꾸고 "
                "GOLD_PATH / ANSWER_PATHS 에 경로를 적어 주세요.")
        print("골드셋 1개와 채점할 답변 파일들을 한꺼번에 고르세요.")
        uploaded = files.upload()
        paths = list(uploaded.keys())
    else:
        paths = ([GOLD_PATH] if GOLD_PATH else []) + list(ANSWER_PATHS)

    if not paths:
        raise RuntimeError("입력 파일이 없습니다.")

    golds = [p for p in paths if _looks_like_goldset(p)]
    answers = [p for p in paths if p not in golds]
    if len(golds) != 1:
        raise RuntimeError(
            "골드셋(questions 배열이 있는 파일)이 정확히 1개여야 합니다. 지금 {}개: {}"
            .format(len(golds), golds))
    if not answers:
        raise RuntimeError("채점할 답변 파일이 없습니다.")
    return golds[0], sorted(answers)


gold_path, answer_paths = collect_inputs()
goldset = load_goldset(gold_path)

print("\n골드셋: {}  ({}문항)".format(os.path.basename(gold_path), len(goldset)))
print("문항 id: {}{}".format(", ".join(goldset.qids[:5]),
                             " ... " + goldset.qids[-1] if len(goldset) > 5 else ""))
print("채점 대상 {}건:".format(len(answer_paths)))
for p in answer_paths:
    print("  - " + os.path.basename(p))

_expected_calls = len(goldset) * len(answer_paths)
print("\nGemini 판정 예상 호출 수: 문항 {} × 제출물 {} = 최대 {}회".format(
    len(goldset), len(answer_paths), _expected_calls))


def _progress_gemini_judge(question, answer, key_facts, citations, retrieved, gold_keys):
    """gemini_judge를 그대로 호출하되, 문항마다 진행 상황을 한 줄씩 찍는다."""
    result = gemini_judge(question, answer, key_facts, citations, retrieved, gold_keys)
    print("  [{}/{}] 판정 완료 (성공 {} / 대체 {})".format(
        judge_stats["calls"], _expected_calls, judge_stats["gemini_ok"], judge_stats["fallback"]))
    return result


outcome = evaluate_submissions(goldset, answer_paths, judge_fn=_progress_gemini_judge)

# BLIND01~05 구성 검사는 "5팀 익명 제출을 만드는 중"일 때만 막아야 한다. 개발·연습
# 단계(자기 팀 답변 1개만 채점해 보는 것)까지 여기서 막으면 매일 쓰는 워크플로를
# 깨뜨린다 — 정확히 5개를 올렸을 때만 최종 제출 시도로 보고 엄격하게 검사한다.
if len(answer_paths) == 5:
    coverage_issues = check_blind_id_coverage(outcome["results"])
    if coverage_issues:
        print("\n[중단] BLIND01~BLIND05 구성에 문제가 있어 파일을 저장하지 않았습니다:")
        for text in coverage_issues:
            print("   · " + text)
        print("\n입력 파일의 이름·개수를 확인하고(정확히 5개, 각 파일명에 BLIND1~5가 "
              "식별 가능해야 함) 이 셀을 다시 실행하세요.")
        raise RuntimeError("BLIND01~BLIND05 구성 오류로 eval 파일을 생성하지 않았습니다.")
else:
    print("\n[안내] 답변 파일이 5개가 아니라({}개) BLIND01~05 최종 제출 형식 검사는 "
          "건너뜁니다 — 개발·연습 단계로 보고 점수만 계산합니다. 실제 제출 때는 "
          "익명 5팀 답변 파일을 정확히 5개 올려야 합니다.".format(len(answer_paths)))

saved = write_eval_file(outcome, team=TEAM, out_dir=OUT_DIR)

print("\nGemini 판정 요약: 총 {}회 호출 / 성공 {} / 규칙기반 대체 {}".format(
    judge_stats["calls"], judge_stats["gemini_ok"], judge_stats["fallback"]))
if judge_stats["fallback"] > 0:
    print("주의: 대체가 발생했습니다 — 위 사유를 확인하세요. 모든 문항을 Gemini로 "
          "처리해야 하므로, 키·쿼터를 점검하고 가능하면 다시 돌리는 것을 권장합니다.")

print("\n" + "=" * 62)
print("{:<14s} {:>10s}  {:<10s} {:>6s} {:>8s} {:>8s}".format(
    "blind_id", "총점", "status", "MRR", "F1", "판정"))
print("-" * 62)
for row in outcome["results"]:
    detail = outcome["details"].get(row["blind_id"], {})
    obj = detail.get("objective", {})
    print("{:<14s} {:>10.4f}  {:<10s} {:>6} {:>8} {:>8}".format(
        str(row["blind_id"]), row["total"], row["status"],
        "{:.3f}".format(obj["mrr"]) if obj else "-",
        "{:.3f}".format(obj["keyfact_f1"]) if obj else "-",
        "{:.1f}".format(detail["judge"]["score_0_100"]) if detail.get("judge") else "-"))
print("=" * 62)
print("저장: " + saved)

# 파일에 문제가 있었던 제출물은 따로 알려 준다 — 제출 파일 자체에는 안 들어가는 진단 정보
for blind_id, detail in outcome["details"].items():
    problems = detail.get("problems") or []
    if problems:
        print("\n[{}] 파일 문제 {}건 (해당 후보는 그래도 completed로 채점됨)".format(blind_id, len(problems)))
        for text in problems[:5]:
            print("   · " + text)

# 운영진 "교차 평가 결과 파일 검사기"와 같은 기준으로 방금 저장한 파일을 바로 검사한다.
# 답변 파일이 5개가 아니면(개발·연습 단계) 이 검사는 당연히 [확인 필요]가 뜬다 —
# BLIND01~05 5개가 아니니 정상이다. 그런 상황이면 결과가 아니라 이유를 알려 준다.
print("\n" + "=" * 62)
if len(answer_paths) != 5:
    print("[참고] 답변 파일이 5개가 아니므로 제출 형식 검사는 생략합니다 "
          "(개발·연습 단계에서는 정상). 위 표의 점수만 확인하면 됩니다.")
else:
    _submit_ok, _submit_issues = check_eval_file(saved)
    if _submit_ok:
        print("[제출 가능] {} — BLIND01~BLIND05가 모두 정상 완료됐습니다.".format(os.path.basename(saved)))
    else:
        print("[확인 필요] {} — 아래 문제를 해결한 뒤 이 셀을 다시 실행하세요.".format(os.path.basename(saved)))
        for text in _submit_issues:
            print("   · " + text)